# Project 2 - Global Space Economy Analysis

## Notebook 02: Collect Financial Data

**Objectives**

- Filter public companies
- Download financial and market data
- Normalize currencies
- Create a financial dataset
- Store the dataset in SQLite

In [15]:
import pandas as pd
import sqlite3
import yfinance as yf

In [16]:
conn = sqlite3.connect("../data/space_economy.db")

In [17]:
public_companies = pd.read_sql(
    """
    SELECT *
    FROM companies
    WHERE `Listing Status` = 'Public';
    """,
    conn
)

public_companies.head()

,Company,Ticker,Country,Listing Status,Sector,Subsector
0,SpaceX,SPCX,US,Public,Infrastructure,Launch
1,Rocket Lab,RKLB,US,Public,Infrastructure,Launch
2,MDA Space,MDA.TO,CA,Public,Infrastructure,Satellite Manufacturing
3,Airbus Defence and Space,AIR.PA,FR,Public,Infrastructure,Satellite Manufacturing
4,中国卫星,600118.SS,CN,Public,Infrastructure,Satellite Manufacturing


In [18]:
rocketlab = yf.Ticker("RKLB")

In [19]:
rocketlab.info

{'address1': '3881 McGowen Street',
 'city': 'Long Beach',
 'state': 'CA',
 'zip': '90808',
 'country': 'United States',
 'phone': '714 465 5737',
 'website': 'https://rocketlabcorp.com',
 'industry': 'Aerospace & Defense',
 'industryKey': 'aerospace-defense',
 'industryDisp': 'Aerospace & Defense',
 'sector': 'Industrials',
 'sectorKey': 'industrials',
 'sectorDisp': 'Industrials',
 'longBusinessSummary': 'Rocket Lab Corporation, a space company, provides launch services and space systems solutions in the United States, Canada, Japan, and internationally. The company operates through launch services and space systems segments. The company provides launch services, spacecraft design services, spacecraft components, spacecraft manufacturing, optical systems, and other spacecraft and on-orbit management solutions and constellation management services, as well as designs and manufactures small and medium-class rockets and develops flight and ground software. It also designs, manufactures,

In [20]:
rocketlab_info = rocketlab.info

In [21]:
rocketlab_data = {
    "Company": "Rocket Lab",
    "Ticker": rocketlab_info.get("symbol"),
    "Market Cap": rocketlab_info.get("marketCap"),
    "Current Price": rocketlab_info.get("currentPrice"),
    "Beta": rocketlab_info.get("beta"),
    "Trailing PE": rocketlab_info.get("trailingPE"),
    "Forward PE": rocketlab_info.get("forwardPE"),
    "Revenue": rocketlab_info.get("totalRevenue"),
    "Net Income": rocketlab_info.get("netIncomeToCommon"),
    "Profit Margin": rocketlab_info.get("profitMargins"),
    "52 Week High": rocketlab_info.get("fiftyTwoWeekHigh"),
    "52 Week Low": rocketlab_info.get("fiftyTwoWeekLow")
}

In [22]:
rocketlab_data

{'Company': 'Rocket Lab',
 'Ticker': 'RKLB',
 'Market Cap': 50635313152,
 'Current Price': 81.04,
 'Beta': 2.553,
 'Trailing PE': None,
 'Forward PE': 2431.443,
 'Revenue': 679577984,
 'Net Income': -182615008,
 'Profit Margin': -0.26872,
 '52 Week High': 151.0,
 '52 Week Low': 37.57}

In [23]:
rocketlab_df = pd.DataFrame([rocketlab_data])

rocketlab_df

,Company,Ticker,Market Cap,Current Price,Beta,Trailing PE,Forward PE,Revenue,Net Income,Profit Margin,52 Week High,52 Week Low
0,Rocket Lab,RKLB,50635313152,81.04,2.553,None,2431.443,679577984,-182615008,-0.26872,151.0,37.57


In [50]:
from datetime import datetime

financials = []

for index, row in public_companies.iterrows():
    company_name = row["Company"]
    ticker_symbol = row["Ticker"]

    try:
        stock = yf.Ticker(ticker_symbol)
        info = stock.info

        # Prevent invalid tickers from being mistakenly identified as successful downloads.
        if not info.get("marketCap") and not info.get("currentPrice"):
            raise ValueError("No valid market data found")

        company_data = {
            "Company": company_name,
            "Ticker": ticker_symbol,
            "Currency": info.get("currency"),

            # Amount in local currency
            "Market Cap Local": info.get("marketCap"),
            "Revenue Local": info.get("totalRevenue"),
            "Net Income Local": info.get("netIncomeToCommon"),

            # Price data, retained in local currency.
            "Current Price Local": info.get("currentPrice"),
            "52 Week High Local": info.get("fiftyTwoWeekHigh"),
            "52 Week Low Local": info.get("fiftyTwoWeekLow"),

            # No currency unit, no conversion required.
            "Beta": info.get("beta"),
            "Trailing PE": info.get("trailingPE"),
            "Forward PE": info.get("forwardPE"),
            "Price to Sales": info.get("priceToSalesTrailing12Months"),
            "Profit Margin": info.get("profitMargins"),
            "Revenue Growth": info.get("revenueGrowth"),

            # Data scraping date
            "Data Date": datetime.now().date().isoformat()
        }

        financials.append(company_data)
        print(f"Downloaded: {company_name}")

    except Exception as e:
        print(f"Failed: {company_name} - {e}")

Downloaded: SpaceX
Downloaded: Rocket Lab
Downloaded: MDA Space
Downloaded: Airbus Defence and Space
Downloaded: 中国卫星
Downloaded: Gilat Satellite Networks
Downloaded: Comtech Telecommunications
Downloaded: Kratos Defense & Security Solutions
Downloaded: Kongsberg Defence & Aerospace
Downloaded: Redwire
Downloaded: Voyager Technologies
Downloaded: Iridium Communications
Downloaded: Viasat
Downloaded: SES
Downloaded: Eutelsat
Downloaded: EchoStar
Downloaded: Globalstar
Downloaded: Telesat
Downloaded: AST SpaceMobile
Downloaded: Planet Labs
Downloaded: BlackSky
Downloaded: Spire Global
Downloaded: Satellogic
Downloaded: Trimble
Downloaded: Hexagon
Downloaded: Fugro
Downloaded: 中科星图
Downloaded: NVIDIA
Downloaded: AMD
Downloaded: Broadcom
Downloaded: Marvell Technology
Downloaded: Microchip Technology
Downloaded: Amazon
Downloaded: Microsoft
Downloaded: Alphabet
Downloaded: Oracle
Downloaded: Palantir
Downloaded: C3.ai
Downloaded: Honeywell
Downloaded: Moog
Downloaded: Parker Hannifin
Downl

In [51]:
financials_df = pd.DataFrame(financials)

financials_df.head()

,Company,Ticker,Currency,Market Cap Local,Revenue Local,Net Income Local,Current Price Local,52 Week High Local,52 Week Low Local,Beta,Trailing PE,Forward PE,Price to Sales,Profit Margin,Revenue Growth,Data Date
0,SpaceX,SPCX,USD,1914209435648,19300999168,-9356000256,145.30,225.64,145.07,NaN,NaN,167.635790,99.176704,-0.44998,0.154,2026-07-12
1,Rocket Lab,RKLB,USD,50635313152,679577984,-182615008,81.04,151.00,37.57,2.553,NaN,2431.443000,74.509940,-0.26872,0.635,2026-07-12
2,MDA Space,MDA.TO,CAD,6691734528,2098800000,47100000,48.17,67.90,20.85,NaN,60.974678,27.616928,3.188362,0.02244,0.002,2026-07-12
3,Airbus Defence and Space,AIR.PA,EUR,155288616960,72529002496,5014000128,197.26,221.30,157.42,0.880,31.162716,22.960196,2.141055,0.06913,-0.066,2026-07-12
4,中国卫星,600118.SS,CNY,106447667200,6270337536,16975960,90.02,127.77,27.27,0.546,9002.000000,630.833860,16.976385,0.00271,0.379,2026-07-12


In [52]:
financials_df.shape

(56, 16)

In [53]:
financials_df["Currency"].value_counts(dropna=False)

Currency
USD    43
EUR     5
CNY     2
SEK     2
CAD     1
NOK     1
GBp     1
KRW     1
Name: count, dtype: int64

In [54]:
def get_usd_rate(currency):
    if pd.isna(currency):
        return None

    if currency == "USD":
        return 1.0

    fx_ticker = f"{currency}USD=X"

    try:
        fx_data = yf.Ticker(fx_ticker).history(period="5d")

        if fx_data.empty:
            return None

        return fx_data["Close"].dropna().iloc[-1]

    except Exception:
        return None

In [55]:
financials_df["USD Exchange Rate"] = (
    financials_df["Currency"].apply(get_usd_rate)
)

In [56]:
financials_df["FX Date"] = datetime.now().date().isoformat()

In [57]:
financials_df[
    financials_df["USD Exchange Rate"].isna()
][["Company", "Ticker", "Currency"]]

,Company,Ticker,Currency


In [58]:
financials_df["Market Cap USD"] = (
    financials_df["Market Cap Local"]
    * financials_df["USD Exchange Rate"]
)

financials_df["Revenue USD"] = (
    financials_df["Revenue Local"]
    * financials_df["USD Exchange Rate"]
)

financials_df["Net Income USD"] = (
    financials_df["Net Income Local"]
    * financials_df["USD Exchange Rate"]
)

In [59]:
financials_df[
    [
        "Company",
        "Currency",
        "USD Exchange Rate",
        "Market Cap Local",
        "Market Cap USD",
        "Revenue USD",
        "Net Income USD",
        "Profit Margin"
    ]
].head(10)

,Company,Currency,USD Exchange Rate,Market Cap Local,Market Cap USD,Revenue USD,Net Income USD,Profit Margin
0,SpaceX,USD,1.000000,1914209435648,1.914209e+12,1.930100e+10,-9.356000e+09,-0.44998
1,Rocket Lab,USD,1.000000,50635313152,5.063531e+10,6.795780e+08,-1.826150e+08,-0.26872
2,MDA Space,CAD,0.706564,6691734528,4.728138e+09,1.482936e+09,3.327916e+07,0.02244
3,Airbus Defence and Space,EUR,1.141944,155288616960,1.773308e+11,8.282403e+10,5.725705e+09,0.06913
4,中国卫星,CNY,0.147783,106447667200,1.573111e+10,9.266464e+08,2.508750e+06,0.00271
5,Gilat Satellite Networks,USD,1.000000,906406080,9.064061e+08,4.700940e+08,3.195300e+07,0.06797
6,Comtech Telecommunications,USD,1.000000,57226336,5.722634e+07,4.541640e+08,-6.576700e+07,-0.05517
7,Kratos Defense & Security Solutions,USD,1.000000,9036588032,9.036588e+09,1.415200e+09,2.940000e+07,0.02077
8,Kongsberg Defence & Aerospace,NOK,0.102351,263442956288,2.696365e+10,3.425688e+09,4.899542e+08,0.21993
9,Redwire,USD,1.000000,2431241984,2.431242e+09,3.709580e+08,-3.438640e+08,-0.80900


In [60]:
financials_df[
    ["Company", "Market Cap USD"]
].sort_values(
    by="Market Cap USD",
    ascending=False
).head(10)

,Company,Market Cap USD
27,NVIDIA,5.109662e+12
34,Alphabet,4.358515e+12
33,Microsoft,2.860690e+12
32,Amazon,2.639149e+12
0,SpaceX,1.914209e+12
29,Broadcom,1.902889e+12
28,AMD,9.096958e+11
35,Oracle,4.051094e+11
36,Palantir,3.039552e+11
50,RTX,2.638557e+11


In [61]:
financials_df.to_sql(
    "financials",
    conn,
    if_exists="replace",
    index=False
)

56

In [62]:
pd.read_sql(
    """
    SELECT
        Company,
        Currency,
        `Market Cap Local`,
        `Market Cap USD`
    FROM financials
    ORDER BY `Market Cap USD` DESC
    LIMIT 10;
    """,
    conn
)

,Company,Currency,Market Cap Local,Market Cap USD
0,NVIDIA,USD,5109662089216,5.109662e+12
1,Alphabet,USD,4358514933760,4.358515e+12
2,Microsoft,USD,2860690440192,2.860690e+12
3,Amazon,USD,2639149400064,2.639149e+12
4,SpaceX,USD,1914209435648,1.914209e+12
5,Broadcom,USD,1902889402368,1.902889e+12
6,AMD,USD,909695778816,9.096958e+11
7,Oracle,USD,405109440512,4.051094e+11
8,Palantir,USD,303955181568,3.039552e+11
9,RTX,USD,263855669248,2.638557e+11
